# Import packages

In [1]:
import re
import os
import pandas as pd
import plotly.express as px

# Custom functions

In [3]:
from tools.obtain_gene_seq import gene_name_to_download_seq
from tools.fimo_motif_scanning import motif_scanning, filter_fimo_output_by_tf_list, merge_rank_by_count_best_q

# Obtain sequence with interested region

In [4]:
seq_direct = gene_name_to_download_seq(
        gene_name = 'MITF',
        seq_type ='promoter',
        upstream_bp = 5000,
        downstream_bp = 2000,
        save_file = True
    )

Sequence is saved to input/obtained_seq/seq_MITF_promoter_5000_2000.fna


# fimo motif scanning

In [5]:
! fimo --version # Run this line to ensure your interpreter can find the finmo software

/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
5.5.8


In [6]:
p_threshold = 1E-4
motif_database_direct = 'input/jaspar_dataset/JASPAR2026_CORE_vertebrates_non-redundant_pfms_meme_original.txt'  # This is the motif database you downloaded from JASPAR

In [7]:
direct_fimo_output = f'output/MITF_jaspar_human_full/fimo_output'
direct_seq_input = f'input/obtained_seq/seq_MITF_promoter_5000_2000.fna'

os.makedirs(direct_fimo_output, exist_ok=True)

result_direct = motif_scanning(
    direct_fimo_output,
    motif_database_direct,
    direct_seq_input,
    p_threshold
)

Running command: fimo --oc output/MITF_jaspar_human_full/fimo_output --verbosity 1 --bgfile --nrdb-- --thresh 0.0001 --no-pgc input/jaspar_dataset/JASPAR2026_CORE_vertebrates_non-redundant_pfms_meme_original.txt input/obtained_seq/seq_MITF_promoter_5000_2000.fna
Motif scanning result saved as output/MITF_jaspar_human_full/fimo_output/fimo.tsv


# Result analyses

## Read in result file

In [8]:
df_gene = pd.read_csv(result_direct, sep='\t')
df_gene.head()

,motif_id,motif_alt_id,sequence_name,start,stop,strand,score,p-value,q-value,matched_sequence
0,MA0088.3,ZNF143,NIH,4844.0,4866.0,+,27.3770,5.880000e-10,0.000008,CCTGGCCTTCTGGGAGCTGTAGT
1,MA1723.2,PRDM9,NIH,6092.0,6111.0,-,21.4184,2.020000e-08,0.000270,AGTGGGAATGGAGGCAGCAA
2,MA1573.2,Thap11,NIH,4853.0,4866.0,-,21.0734,4.290000e-08,0.000596,ACTACAGCTCCCAG
3,MA1723.2,PRDM9,NIH,4750.0,4769.0,+,19.6429,1.230000e-07,0.000820,AGTGGGGAGAGAGGAGGGAG
4,MA2546.1,ZNF131,NIH,5447.0,5459.0,-,19.0284,1.790000e-07,0.002410,CCGCGCCCCCGCG


In [10]:
df_gene['TF'] = df_gene['motif_alt_id'].fillna(df_gene['motif_id'])

df_gene['TF_stripped'] = df_gene['TF'].apply(
    lambda name: re.sub(r'[^A-Za-z0-9]', '', name))  #Create a new col for stripped TF name
df_gene

,motif_id,motif_alt_id,sequence_name,start,stop,strand,score,p-value,q-value,matched_sequence,TF,TF_stripped
0,MA0088.3,ZNF143,NIH,4844.0,4866.0,+,27.377000,5.880000e-10,0.000008,CCTGGCCTTCTGGGAGCTGTAGT,ZNF143,ZNF143
1,MA1723.2,PRDM9,NIH,6092.0,6111.0,-,21.418400,2.020000e-08,0.000270,AGTGGGAATGGAGGCAGCAA,PRDM9,PRDM9
2,MA1573.2,Thap11,NIH,4853.0,4866.0,-,21.073400,4.290000e-08,0.000596,ACTACAGCTCCCAG,Thap11,Thap11
3,MA1723.2,PRDM9,NIH,4750.0,4769.0,+,19.642900,1.230000e-07,0.000820,AGTGGGGAGAGAGGAGGGAG,PRDM9,PRDM9
4,MA2546.1,ZNF131,NIH,5447.0,5459.0,-,19.028400,1.790000e-07,0.002410,CCGCGCCCCCGCG,ZNF131,ZNF131
...,...,...,...,...,...,...,...,...,...,...,...,...
1729,MA2551.1,ZNF286B,NIH,2849.0,2869.0,+,-0.134146,9.990000e-05,0.464000,AGGTCTGGAAGGTCACTATTT,ZNF286B,ZNF286B
1730,MA0144.3,STAT3,NIH,6558.0,6566.0,+,10.326500,9.990000e-05,0.170000,ATCTGGGAA,STAT3,STAT3
1731,# FIMO (Find Individual Motif Occurrences): Ve...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,# FIMO (Find Individual Motif Occurrences): Ve...,FIMOFindIndividualMotifOccurrencesVersion558co...
1732,# The format of this file is described at http...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,# The format of this file is described at http...,Theformatofthisfileisdescribedathttpsmemesuite...


## Filter df by TF names

In [11]:
print(f'>>>> Processing fimo output for MITF')

tf_pkl_direct = f'input/filters/tf_mbm.pkl'

fimo_df_filtered = filter_fimo_output_by_tf_list(tf_pkl_direct, df_gene)

>>>> Processing fimo output for MITF
Before filtering, df has a shape as (1734, 12)
After filtering, df has a shape as (969, 12)


## Generate & save merged dataset

In [12]:
df_save_direct = f'output/MITF_jaspar_human_full/MITF_mbm_filter_hit_tf_wise.csv'

merged_df = merge_rank_by_count_best_q(fimo_df_filtered, df_save_direct)

## Plot binding quality vs. number of binding sites

In [13]:
fig = px.scatter(
    merged_df,
    x="site_count",
    y="best_q_per_tf",
    hover_name="TF",  # big bold line in tooltip
    hover_data={
        "site_count": True,  # show extra fields in tooltip
        "best_q_per_tf": True,
        "TF": False
    },  # don't duplicate TF in the table
    labels={
        "site_count": "Sites per TF",
        "best_q_per_tf": "Best q score per TF",
    },
    title='MITF in MBM cell lines: TF binding quality vs. binding site count',
    width=800,
    height=800
)

# nicer layout + grid
fig.update_layout(template="plotly_white")

fig.show()
fig.write_html(f'output/MITF_jaspar_human_full/MITF_MBM_filter_hit_tf.html')